# GRASP Library Designer (Colab)

Redesign and anneal the **42-module combinatorial GRASP library** (Farley et al., *NAR* 2025), then compile a target RNA with GAP.

Cut indices stay fixed; only synonymous DNA is redesigned. Objectives: ligation fidelity, codon optimality, synthesis fitness.

**Colab tip:** each cell uses **Forms**. Titles use `{display-mode: "form"}` so code stays hidden by default (cell ⋮ → **Form → Hide code**).

For a **single RNA → free cut sites** design (no library modules), open `grasp_oneshot_designer.ipynb`.


In [ ]:
#@title 0 · Install { display-mode: "form" }
#@markdown Clone the private repo (leave token empty if the notebook already runs from the repo root).

GITHUB_TOKEN = "" #@param {type:"string"}
REPO_SLUG = "JustABiologist/grasp-library-designer" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

if IN_COLAB:
    repo_dir = Path("/content/grasp-library-designer")
    if not (repo_dir / "grasp_library").is_dir():
        if not REPO_SLUG or "OWNER/" in REPO_SLUG:
            raise ValueError("Set REPO_SLUG to your GitHub owner/repo (private).")
        auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN.strip() else ""
        url = f"https://{auth}github.com/{REPO_SLUG}.git"
        print(f"Cloning github.com/{REPO_SLUG} @ {BRANCH} …")
        subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(repo_dir)])
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
    _pip("-e", ".")
    print("Installed from", repo_dir)
else:
    root = Path.cwd()
    if not (root / "grasp_library").is_dir():
        for p in [root, *root.parents]:
            if (p / "grasp_library").is_dir():
                os.chdir(p)
                sys.path.insert(0, str(p))
                break
    _pip("-e", ".")
    print("Using local package at", Path.cwd())

print("Ready.")



In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown Organism, synthesis, ligation, architecture, and anneal depth — then run this cell.

target_rna = "UUACACGUG" #@param {type:"string"}
organism = "Escherichia coli (Kazusa)" #@param ["Escherichia coli (Kazusa)", "Saccharomyces cerevisiae (Kazusa)", "Homo sapiens (Kazusa)", "Euglena gracilis nuclear (Kazusa)", "Chlamydomonas reinhardtii nuclear (Kazusa)", "Chlamydomonas reinhardtii chloroplast (Kazusa)"]
genetic_code = 1 #@param {type:"integer"}
nterm_overhang = "AGGT" #@param ["AGGT", "AATG"]
architecture = "9S" #@param ["9S", "14S", "19S"]
synthesis_vendor = "Twist · Standard gene guidelines" #@param ["Twist · Express / Low complexity", "Twist · Standard gene guidelines", "Twist · Complex Genes tolerant", "IDT · gBlocks / eBlocks conservative", "Generic · conservative (default)"]
assembly_enzyme = "GRASP default · BsaI + BpiI + BsmBI" #@param ["GRASP default · BsaI + BpiI + BsmBI", "BsaI (GGTCTC)", "BpiI / BbsI (GAAGAC)", "BsmBI / Esp3I (CGTCTC)", "None (no enzyme filter)"]
ligation_table = "T4 · 18 h · 25 °C (Potapov)" #@param ["T4 · 18 h · 25 °C (Potapov)", "T4 · 18 h · 37 °C (Potapov)", "T4 · 1 h · 25 °C (Potapov)", "BsaI-HFv2 · constant 37 °C", "BsmBI-v2 · constant 42 °C"]
overhang_redesign = True #@param {type:"boolean"}
redesign_selection = "knee" #@param ["knee", "max_fidelity"]
optimize_depth = 2000 #@param {type:"integer"}

from pathlib import Path
from grasp_library import build_default_config
from grasp_library.colab_forms import apply_form_settings
from grasp_library import notebook_ui as ui

PROJECT_DIR = Path("grasp_library_project")
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
PROFILE_GB = PROJECT_DIR / "profiles" / "grasp_nar2025" / "genbank"
for d in (INPUT_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG = build_default_config(INPUT_DIR)
CONFIG["project_name"] = "GRASP_library_colab"

applied = apply_form_settings(
    CONFIG,
    organism=organism,
    genetic_code=int(genetic_code),
    target_rna=target_rna,
    nterm_overhang=nterm_overhang,
    architecture=architecture,
    synthesis_vendor=synthesis_vendor,
    assembly_enzyme=assembly_enzyme,
    ligation_table=ligation_table,
    overhang_redesign=bool(overhang_redesign),
    redesign_selection=redesign_selection,
    optimize_depth=int(optimize_depth),
)
CONFIG = applied["config"]
CODON_DATA = applied["codon_data"]
CODON_TABLE = applied["codon_table"]
SELECTED_ORGANISM = organism

# Shared run state (cleared when settings change)
parts = None
target_map = None
PARETO_FRONT = None
SELECTED_OVERHANGS = None
parts_with_redesigned_junctions = None
optimized_library = None
PARETO_PLOT = None
ASSEMBLY = None

ui.status(
    f"Target <b>{CONFIG['target_rna']}</b> · <b>{organism}</b> · "
    f"arch <b>{architecture}</b> · N-term <code>{nterm_overhang}</code> · "
    f"redesign <b>{CONFIG['overhang_redesign']['enabled']}</b> ({redesign_selection}) · "
    f"depth <b>{CONFIG['optimizer']['iterations_per_part']:,}</b>"
)



In [ ]:
#@title 2 · Import GRASP modules { display-mode: "form" }
#@markdown Loads Farley et al. NAR 2025 GenBank modules into `grasp_library_project/input/`.

FORCE_REIMPORT = False #@param {type:"boolean"}

from IPython.display import display
from grasp_library import ensure_grasp_imported
from grasp_library import notebook_ui as ui

if not CODON_DATA:
    ui.note("Run Settings first.")
else:
    imported = ensure_grasp_imported(
        profile_genbank_dir=PROFILE_GB,
        input_dir=INPUT_DIR,
        force=bool(FORCE_REIMPORT),
        log=print,
    )
    parts = imported["parts"]
    target_map = imported.get("target_map")
    display(parts.head())
    jm = imported["junction_map"]
    display(
        jm.groupby("junction", as_index=False)
        .first()[["junction", "native_overhang"]]
    )
    ui.status(
        f"<b>{len(parts)}</b> modules · "
        f"<b>{jm['junction'].nunique()}</b> junctions · "
        f"<b>{len(imported['overhang_candidates'])}</b> overhang candidates"
    )



In [ ]:
#@title 3 · Redesign overhangs { display-mode: "form" }
#@markdown Pareto search over synonym overhangs at **fixed** GRASP cut indices. Skip if redesign is off in Settings.

RUN_OVERHANG_REDESIGN = True #@param {type:"boolean"}
SEED = 42 #@param {type:"integer"}

import random
import numpy as np
import pandas as pd
from IPython.display import display
from grasp_library import (
    LigationFidelityCalculator,
    load_and_validate_parts,
    optimize_coding_sequence,
    run_overhang_redesign,
)
from grasp_library import notebook_ui as ui

random.seed(int(SEED))
np.random.seed(int(SEED))

PARETO_FRONT = None
SELECTED_OVERHANGS = None
parts_with_redesigned_junctions = None

if not RUN_OVERHANG_REDESIGN:
    ui.note("RUN_OVERHANG_REDESIGN is off.")
elif not CODON_DATA:
    ui.note("Run Settings first.")
else:
    if parts is None and CONFIG["parts_file"].exists():
        parts = load_and_validate_parts(CONFIG["parts_file"])
    lig = CONFIG["ligation"]
    FIDELITY = LigationFidelityCalculator(
        temperature=lig["temperature"],
        hours=lig["hours"],
        ligation_table=lig.get("ligation_table"),
        min_efficiency=lig.get("min_efficiency", 0.25),
        min_fidelity=lig.get("min_fidelity", 0.9),
    )
    if not CONFIG.get("overhang_redesign", {}).get("enabled", True):
        ui.note("Overhang redesign disabled in Settings — keeping native overhangs.")
        parts_with_redesigned_junctions = parts
    else:
        PARETO_FRONT, SELECTED_OVERHANGS, parts_with_redesigned_junctions = run_overhang_redesign(
            parts=parts,
            codon_data=CODON_DATA,
            config=CONFIG,
            optimize_coding_sequence=optimize_coding_sequence,
            input_dir=INPUT_DIR,
            output_dir=OUTPUT_DIR,
            seed=int(SEED),
            fidelity=FIDELITY,
            log=print,
        )
        display(PARETO_FRONT)
        display(
            pd.DataFrame([SELECTED_OVERHANGS], index=["overhang"])
            .T.rename(columns={"overhang": "selected"})
        )
        ui.status("Overhang redesign done — next: anneal library.")



In [ ]:
#@title 4 · Anneal library { display-mode: "form" }
#@markdown Full CDS simulated annealing for every module (uses redesigned masks when available).

RUN_LIBRARY_OPTIMIZE = True #@param {type:"boolean"}

from IPython.display import display
from grasp_library import (
    load_and_validate_parts,
    optimize_library,
    run_library_optimize,
)
from grasp_library import notebook_ui as ui

optimized_library = None

if not RUN_LIBRARY_OPTIMIZE:
    ui.note("RUN_LIBRARY_OPTIMIZE is off.")
elif not CODON_DATA:
    ui.note("Run Settings first.")
else:
    source_parts = (
        parts_with_redesigned_junctions
        if parts_with_redesigned_junctions is not None
        else parts
    )
    if source_parts is None and CONFIG["parts_file"].exists():
        source_parts = load_and_validate_parts(CONFIG["parts_file"])
    optimized_library = run_library_optimize(
        parts=source_parts,
        codon_data=CODON_DATA,
        config=CONFIG,
        optimize_library=optimize_library,
        output_dir=OUTPUT_DIR,
        log=print,
    )
    if SELECTED_OVERHANGS:
        tag = ";".join(f"{k}={v}" for k, v in sorted(SELECTED_OVERHANGS.items()))
        optimized_library = optimized_library.copy()
        optimized_library["selected_overhangs"] = tag
        optimized_library.to_csv(OUTPUT_DIR / "optimized_library.csv", index=False)
    display(optimized_library.head())
    ui.status(
        f"Annealed <b>{len(optimized_library)}</b> sequences → "
        f"<code>{OUTPUT_DIR / 'optimized_library.csv'}</code>"
    )



In [ ]:
#@title 5 · Pareto plot { display-mode: "form" }
#@markdown Rescores the front after redesign + anneal (codon = full CDS; synthesis = full oligo).

RUN_PARETO_PLOT = True #@param {type:"boolean"}
DEEP_RESCORE_ALL = False #@param {type:"boolean"}
#@markdown `DEEP_RESCORE_ALL` fully anneals every front member (slow).

import pandas as pd
from IPython.display import display
from grasp_library import (
    load_and_validate_parts,
    plot_library_pareto_after_anneal,
)
from grasp_library.workflows import parse_overhang_selection
from grasp_library import notebook_ui as ui

PARETO_PLOT = None

if not RUN_PARETO_PLOT:
    ui.note("RUN_PARETO_PLOT is off.")
elif optimized_library is None or getattr(optimized_library, "empty", False):
    ui.note("Run Anneal library first.")
else:
    front = PARETO_FRONT
    selected = SELECTED_OVERHANGS
    src_parts = parts if parts is not None else None

    if front is None or getattr(front, "empty", True):
        path = OUTPUT_DIR / "pareto_front.csv"
        if path.exists():
            front = pd.read_csv(path)
            PARETO_FRONT = front
        else:
            ui.note("No Pareto front — run Redesign overhangs first (or enable redesign).")
            front = None

    if front is not None and not front.empty:
        if selected is None and (OUTPUT_DIR / "selected_overhangs.csv").exists():
            sel = pd.read_csv(OUTPUT_DIR / "selected_overhangs.csv")
            if "overhangs" in sel.columns:
                selected = parse_overhang_selection(sel.iloc[0]["overhangs"])
                SELECTED_OVERHANGS = selected
        if src_parts is None and CONFIG["parts_file"].exists():
            src_parts = load_and_validate_parts(CONFIG["parts_file"])
            parts = src_parts

        junction_map = pd.read_csv(INPUT_DIR / "junction_map.csv")
        PARETO_PLOT = plot_library_pareto_after_anneal(
            front=front,
            parts=src_parts,
            junction_map=junction_map,
            codon_data=CODON_DATA,
            config=CONFIG,
            optimized_library=optimized_library,
            selected_overhangs=selected,
            fidelity=globals().get("FIDELITY"),
            deep_all=bool(DEEP_RESCORE_ALL),
            output_dir=OUTPUT_DIR,
            log=print,
        )
        display(PARETO_PLOT["front"])
        display(PARETO_PLOT["figure"])
        chosen = PARETO_PLOT["chosen"]
        ui.status(
            f"Selected · synthesis <b>{float(chosen['synthesis']):.3f}</b> · "
            f"codon <b>{float(chosen['codon_optimality']):.3f}</b> · "
            f"fidelity <b>{float(chosen['ligation_fidelity']):.6f}</b><br>"
            f"Plot → <code>{OUTPUT_DIR / 'pareto_front.png'}</code>"
        )



In [ ]:
#@title 6 · Export library { display-mode: "form" }
#@markdown Writes CSV / FASTA / Excel under `grasp_library_project/output/` and downloads the Excel in Colab.

from grasp_library import export_optimized_library
from grasp_library import notebook_ui as ui

if optimized_library is None:
    ui.note("Run Anneal library first.")
else:
    paths = export_optimized_library(
        optimized_library,
        OUTPUT_DIR,
        selected_overhangs=SELECTED_OVERHANGS,
    )
    ui.status(
        "Exported:<br/>"
        f"• <code>{paths['csv']}</code><br/>"
        f"• <code>{paths['fasta']}</code><br/>"
        f"• <code>{paths['xlsx']}</code>"
    )
    if "google.colab" in __import__("sys").modules:
        from google.colab import files
        files.download(str(paths["xlsx"]))
    for p in sorted(OUTPUT_DIR.glob("optimized_grasp_*")):
        if p.is_file():
            print(p.name)



In [ ]:
#@title 7 · Compile target RNA { display-mode: "form" }
#@markdown GAP-compile the Settings target against the annealed library and stitch the CDS.

RUN_COMPILE = True #@param {type:"boolean"}

from IPython.display import display
from grasp_library import compile_and_assemble_target
from grasp_library import notebook_ui as ui

ASSEMBLY = None

if not RUN_COMPILE:
    ui.note("RUN_COMPILE is off.")
elif optimized_library is None:
    ui.note("Run Anneal library first.")
else:
    ASSEMBLY = compile_and_assemble_target(
        target_rna=CONFIG["target_rna"],
        optimized_library=optimized_library,
        config=CONFIG,
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        architecture=CONFIG.get("architecture", "9S"),
        nterm_overhang=CONFIG.get("nterm_overhang", "AGGT"),
    )
    display(ASSEMBLY["assembly_plan"])
    asm = ASSEMBLY["assembled"]
    warn = asm.get("stitch_warning") or ""
    ui.status(
        f"Target <b>{ASSEMBLY['target_rna']}</b> · "
        f"translation verified <b>{asm.get('translation_verified')}</b>"
        + (f"<br/>{warn}" if warn else "")
        + f"<br/>Plan → <code>{ASSEMBLY['plan_csv']}</code>"
        + f"<br/>FASTA → <code>{ASSEMBLY['assembled_fasta']}</code>"
    )
    for key, value in asm.items():
        if key not in {"assembled_cds", "expected_protein", "observed_protein"}:
            print(f"{key}: {value}")



## Notes

| Step | What happens |
|---|---|
| Import | Farley et al. GenBank → `parts.csv`, junction map, overhang candidates |
| Redesign | Synonym overhangs at **fixed** cut indices (Pareto: fidelity / codon / synthesis) |
| Anneal | Masked SA for every module; protein unchanged |
| Pareto plot | Re-score after full oligo synthesis scoring |
| Compile | GAP part order for the target RNA + CDS stitch |

For a **single binder without the combinatorial library**, use `grasp_oneshot_designer.ipynb`.

Hidden form code is only a presentation setting — the source remains in the notebook file.
